<a href="https://colab.research.google.com/github/lokesh018-sys/student-performance-analytics/blob/main/student_performance_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import pandas as pd
from google.colab import files

np.random.seed(42)

print("Student Performance Analytics Engine")
print("Libraries imported successfully!")

Student Performance Analytics Engine
Libraries imported successfully!


In [11]:
NUM_STUDENTS = 500

subjects = [
    "Python",
    "Mathematics",
    "DBMS",
    "Java",
    "Statistics",
    "Data Science"
]

student_ids = [
    f"STU{i:03d}" for i in range(1, NUM_STUDENTS + 1)
]

marks = np.random.normal(
    loc=70,
    scale=12,
    size=(NUM_STUDENTS, len(subjects))
)

marks = np.clip(marks, 0, 100)

marks = np.rint(marks).astype(int)

marks[9, 2] = 5
marks[24, 4] = 100

print("Matrix shape:", marks.shape)
print("Total students:", marks.shape[0])
print("Total subjects:", marks.shape[1])
print("First 5 rows:\n", marks[:5])

Matrix shape: (500, 6)
Total students: 500
Total subjects: 6
First 5 rows:
 [[76 68 78 88 67 67]
 [89 79 64 77 64 64]
 [73 47 49 63 58 74]
 [59 53 88 67 71 53]
 [63 71 56 75 63 66]]


In [12]:
df = pd.DataFrame(marks, columns=subjects)
df.insert(0, "Student_ID", student_ids)

display(df.head(10))

,Student_ID,Python,Mathematics,DBMS,Java,Statistics,Data Science
0,STU001,76,68,78,88,67,67
1,STU002,89,79,64,77,64,64
2,STU003,73,47,49,63,58,74
3,STU004,59,53,88,67,71,53
4,STU005,63,71,56,75,63,66
5,STU006,63,92,70,57,80,55
6,STU007,73,46,54,72,79,72
7,STU008,69,66,52,61,64,83
8,STU009,74,49,74,65,62,77
9,STU010,82,81,5,66,74,82


In [13]:
subject_mean = np.mean(marks, axis=0)
subject_median = np.median(marks, axis=0)
subject_std = np.std(marks, axis=0)
subject_variance = np.var(marks, axis=0)

subject_summary = pd.DataFrame({
    "Subject": subjects,
    "Mean": subject_mean,
    "Median": subject_median,
    "Standard_Deviation": subject_std,
    "Variance": subject_variance,
    "Highest_Mark": np.max(marks, axis=0),
    "Lowest_Mark": np.min(marks, axis=0)
}).round(2)

display(subject_summary)

,Subject,Mean,Median,Standard_Deviation,Variance,Highest_Mark,Lowest_Mark
0,Python,71.04,71.0,11.28,127.30,100,38
1,Mathematics,69.95,70.0,12.08,145.82,100,34
2,DBMS,69.68,70.0,11.93,142.28,100,5
3,Java,70.24,70.0,11.86,140.59,100,35
4,Statistics,69.87,70.0,11.95,142.73,100,31
5,Data Science,71.26,71.0,11.77,138.43,100,36


In [14]:
student_total = np.sum(marks, axis=1)
student_mean = np.mean(marks, axis=1)
student_median = np.median(marks, axis=1)
student_std = np.std(marks, axis=1)

student_results = pd.DataFrame({
    "Student_ID": student_ids,
    "Total_Marks": student_total,
    "Percentage": student_total / len(subjects),
    "Mean": student_mean,
    "Median": student_median,
    "Standard_Deviation": student_std
})

# Rank 1 is the student with the highest total
student_results["Rank"] = (
    student_results["Total_Marks"]
    .rank(method="min", ascending=False)
    .astype(int)
)

student_results = student_results.sort_values(
    by=["Total_Marks", "Student_ID"],
    ascending=[False, True]
).reset_index(drop=True)

display(student_results.head(10).round(2))

,Student_ID,Total_Marks,Percentage,Mean,Median,Standard_Deviation,Rank
0,STU250,522,87.00,87.00,87.0,6.40,1
1,STU195,502,83.67,83.67,81.0,7.65,2
2,STU071,499,83.17,83.17,84.0,9.12,3
3,STU385,496,82.67,82.67,79.0,10.98,4
4,STU080,491,81.83,81.83,80.0,10.82,5
5,STU430,491,81.83,81.83,85.0,13.83,5
6,STU187,490,81.67,81.67,83.0,9.69,7
7,STU451,490,81.67,81.67,82.5,6.77,7
8,STU149,483,80.50,80.50,81.5,3.10,9
9,STU063,478,79.67,79.67,81.0,11.06,10


In [15]:
subject_mean = np.mean(marks, axis=0)
subject_std = np.std(marks, axis=0)

safe_std = np.where(subject_std == 0, 1, subject_std)

z_scores = (marks - subject_mean) / safe_std

outlier_mask = np.abs(z_scores) > 2.5

student_indices, subject_indices = np.where(outlier_mask)

outliers = pd.DataFrame({
    "Student_ID": [student_ids[i] for i in student_indices],
    "Subject": [subjects[j] for j in subject_indices],
    "Mark": marks[student_indices, subject_indices],
    "Z_Score": z_scores[student_indices, subject_indices]
}).round(2)

print("Total unusual marks detected:", len(outliers))
display(outliers.head(20))

Total unusual marks detected: 33


,Student_ID,Subject,Mark,Z_Score
0,STU010,DBMS,5,-5.42
1,STU013,DBMS,39,-2.57
2,STU025,Statistics,100,2.52
3,STU044,Statistics,31,-3.25
4,STU080,Statistics,100,2.52
5,STU091,Statistics,40,-2.50
6,STU108,Statistics,38,-2.67
7,STU110,Python,100,2.57
8,STU112,DBMS,38,-2.66
9,STU128,Python,100,2.57


In [16]:
df["Total_Marks"] = student_total
df["Percentage"] = student_total / len(subjects)
df["Rank"] = (
    student_results.set_index("Student_ID")["Rank"]
    .reindex(student_ids).to_numpy()
)

df.to_csv("student_results.csv", index=False)
subject_summary.to_csv("subject_summary.csv", index=False)
outliers.to_csv("outliers.csv", index=False)

print("All CSV reports created successfully!")

All CSV reports created successfully!


In [17]:
files.download("student_results.csv")
files.download("subject_summary.csv")
files.download("outliers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>